## Input and output locations

This notebook uses the full sample only; the old low/high R&D intensity subsample
paths and selector are no longer used. Own R&D intensity (`int_propre`) is retained
for specification (5).

- Input: `RESULTS_DIR/regression_panels/panel_{BASE}_F.csv`, produced by `panel_setup`.
- Output: `RESULTS_DIR/estimation_results/{BASE}/tab_{BASE}_{METRIC}{CHOC_TAG}{OUT_TAG}.xlsx`.
  The reference notebook has no `OUT_TAG`; the other notebooks keep their existing
  `_finetune` and `_scaled` suffixes.


In [1]:
import os
import pandas as pd
import numpy as np
import contextlib
import io
from linearmodels.panel import PanelOLS
import importlib
import project_settings
importlib.reload(project_settings)
from project_settings import PATH_WRK, BASE, METRIC_WANTED, METRIC, RESULTS_DIR, RUN_DIR, CHOC, CHOC_SCENARIO, CHOC_TAG
print(f"CHOC = {CHOC}" + (f" (scenario {CHOC_SCENARIO})" if CHOC else "") + f" -> {RESULTS_DIR}")


CHOC = False -> D:/ENTICE/runs/SRDQ (c,s)/


In [2]:
# ============================================================
# What each metric means:
# ============================================================
#
# --- Metric prefixes:
#     "embod" -> Embodied: indirect R&D content transmitted through
#                intermediate input-output flows.
#     "dir"   -> Direct: direct R&D content transmitted.
#
# ---  AGGREGATED VARIABLES ---
#
#   rd_embod / rd_dir:
#       Total R&D content (embodied across all trade flows for MRIO bases,
#       or direct knowledge transmitted via patent citations).
#
#   rd_embod_D / rd_dir_D:
#       Domestic R&D component — R&D originating from within the same country/region.
#
#   rd_embod_F / rd_dir_F:
#       Foreign R&D component — R&D originating from foreign countries/regions.
#
#   rd_embod_I / rd_dir_I:
#       Intra-sectoral R&D component — R&D originating from the same sector.
#
#   rd_embod_X / ln_rd_dir_X:
#       Inter-sectoral R&D component — R&D originating from all other (different) industries.
#
#
# --- KNOWLEDGE EXTERNALITIES (Country x Sector Interaction Matrix) ---
#
#   rd_same_sector_same_country:
#       Domestic Intra-sectoral spillovers — R&D originating from the same sector within the same country.
#
#   rd_same_sector_diff_country:
#       Foreign Intra-sectoral spillovers — R&D originating from the same sector in foreign countries.
#
#   rd_diff_sector_same_country:
#       Domestic Inter-sectoral spillovers — R&D originating from different sectors within the same country.
#
#   rd_diff_sector_diff_country:
#       Foreign Inter-sectoral spillovers — R&D originating from different sectors in foreign countries.

In [3]:
dep_var = "ln_c_u"  # Dependent variable: log of unit cost


def panel_path():
    """Return the final full-sample panel produced by panel_setup."""
    return RESULTS_DIR + f"regression_panels/panel_{BASE}_F.csv"


def xlsx_path_for():
    return RESULTS_DIR + f"estimation_results/{BASE}/tab_{BASE}_{METRIC}{CHOC_TAG}.xlsx"   # {CHOC_TAG}: "" (no shock) or "_chocB" / "_chocA"


def load_panel():
    """Read the final full-sample panel and add the log-transformed variables."""
    df_init = pd.read_csv(panel_path(), sep=",")
    df_init = df_init.sort_values(["Region", "Sector", "annee"])

    # ====== Log-transformation of all the variables ======
    df_init["ln_c_u"] = np.log(df_init["c_u"].replace(0, np.nan))  # log of unit cost
    df_init["ln_w_l"] = np.log(df_init["w_l"].replace(0, np.nan))  # log of wage rate
    df_init["ln_C_k"] = np.log(df_init["C_k"].replace(0, np.nan))  # log of cost of capital

    # EXIOBASE, FIGARO, OECD_ICIO, GTAP, CAPITAL
    if BASE in ["FIGARO", "EXIOBASE", "OECD_ICIO", "GTAP", "CAPITAL"]:
        df_init[f"ln_rd_{METRIC_WANTED}"] = np.log(df_init[f"rd_{METRIC_WANTED}"].replace(0, np.nan))
        df_init[f"ln_rd_{METRIC_WANTED}_D"] = np.log(df_init[f"rd_{METRIC_WANTED}_D"].replace(0, np.nan))
        df_init[f"ln_rd_{METRIC_WANTED}_F"] = np.log(df_init[f"rd_{METRIC_WANTED}_F"].replace(0, np.nan))
        df_init[f"ln_rd_{METRIC_WANTED}_X"] = np.log(df_init[f"rd_{METRIC_WANTED}_X"].replace(0, np.nan))
        df_init[f"ln_rd_{METRIC_WANTED}_I"] = np.log(df_init[f"rd_{METRIC_WANTED}_I"].replace(0, np.nan))

    # Patents_Citations, FDI
    if BASE in ["Patents_Citation", "FDI"]:
        df_init["ln_rd_dir"] = np.log(df_init["rd_dir"].replace(0, np.nan))
        df_init["ln_rd_dir_D"] = np.log(df_init["rd_dir_D"].replace(0, np.nan))
        df_init["ln_rd_dir_F"] = np.log(df_init["rd_dir_F"].replace(0, np.nan))
        df_init["ln_rd_dir_X"] = np.log(df_init["rd_dir_X"].replace(0, np.nan))
        df_init["ln_rd_dir_I"] = np.log(df_init["rd_dir_I"].replace(0, np.nan))

    for block in ["same_sector_same_country", "same_sector_diff_country",
                  "diff_sector_same_country", "diff_sector_diff_country"]:
        df_init[f"ln_rd_{block}_{METRIC}"] = np.log(df_init[f"rd_{block}_{METRIC}"].replace(0, np.nan))

    # interaction: knowledge stock x own R&D intensity of the sector, (R/Y) * ln(X)   (intensity in %)
    if "int_propre" in df_init.columns:
        ln_total = f"ln_rd_{METRIC_WANTED}"
        df_init[f"int_x_{ln_total}_propre"] = df_init["int_propre"] * df_init[ln_total]
        df_init[f"int_x_{ln_total}"] = df_init["intensité"] * df_init[ln_total]

    return df_init


In [4]:
def estimation(data, dep_var, EXP_VAR):
    """
    Estimates a unit cost function under linear homogeneity in factor prices
    (alpha + (1 - alpha) = 1) using within Entity Fixed Effects (PanelOLS).

    Econometric specification:
        ln(c_u) - ln(w_l) = alpha * [ln(C_k) - ln(w_l)] + gamma * R&D_vars + beta * Controls + mu_i + eps_it

    Parameters
    ----------
    data : pd.DataFrame
        Panel dataframe indexed by MultiIndex (Region, annee) for a given sector.
    dep_var : str
        Name of the dependent variable (e.g., 'ln_c_u').
    EXP_VAR : list
        List of explanatory variables including factor prices, R&D metrics, and controls.

    Returns
    -------
    res : linearmodels.panel.results.PanelResults
        Fitted PanelOLS regression results object.
    alpha : float
        Estimated capital cost share (coefficient on C_k - w_l).
    """
    reg_vars = [dep_var, "ln_C_k", "ln_w_l"] + [c for c in EXP_VAR if c not in ["ln_C_k", "ln_w_l"]]
    sub = data[reg_vars].copy()

    sub = sub.replace([np.inf, -np.inf], np.nan).dropna()

    n_entities = sub.index.get_level_values("Region").nunique()
    n_obs = len(sub)
    print(f"Observations utilisables : {n_obs} | Régions : {n_entities}")

    if n_obs < 30 or n_entities < 3:
        print(">> Trop peu d'observations, estimation annulée.")
        return None, None

    sub["y_adj"] = sub[dep_var] - sub["ln_w_l"]
    sub["C_k-w_l"] = sub["ln_C_k"] - sub["ln_w_l"]

    other_vars = [c for c in EXP_VAR if c not in ["ln_C_k", "ln_w_l"]]

    y = sub["y_adj"]
    valid_x = ["C_k-w_l"] + [c for c in other_vars if sub[c].nunique() > 1]

    if not valid_x:
        print(">> Aucun régresseur valide (variance nulle), estimation annulée.")
        return None, None          # <-- ajouté aussi ici par cohérence

    X = sub[valid_x].assign(const=1.0)
    mod = PanelOLS(y, X, entity_effects=True, time_effects=False, drop_absorbed=True, check_rank=False)
    res = mod.fit(cov_type="clustered", cluster_entity=True, debiased=True)

    alpha = res.params["C_k-w_l"]

    print("\n--- Résultats estimation ---")
    res_df = pd.DataFrame(
        {
            "coef": res.params,
            "std_err": res.std_errors,
            "t_stat": res.tstats,
            "p_value": res.pvalues,
        }).round(4)
    print(res_df)

    print(f"R2 (within) = {res.rsquared_within:.4f} | N = {res.nobs}")

    return res, alpha

Export

In [5]:
def _stars(coef, pvalue):
    """+++/++/+ if positive and significant at 1%/5%/10%, ---/--/- if negative, ‘ns’ otherwise."""
    if pd.isna(coef) or pd.isna(pvalue):
        return "ns"
    if pvalue < 0.01:
        n = 3
    elif pvalue < 0.05:
        n = 2
    elif pvalue < 0.10:
        n = 1
    else:
        return "ns"
    return ("+" if coef > 0 else "-") * n


def _fmt_cell(coef, pvalue, decimals=4):
    if coef is None or pd.isna(coef):
        return "-"
    return f"{coef:.{decimals}f} ({_stars(coef, pvalue)})"


def export_estimation_table(df_init, models, dep_var="ln_c_u", controls=None,
                             xlsx_path="tab_results.xlsx", decimals=4):
    """Runs estimation() for each sector and each model, and directly exports
    a summary table in XLSX format (one row per sector), in ‘coef (stars)’ format
    -- without displaying anything in the console.

    models: ordered dictionary {model_name: {display_name: ln_column}}, e.g. for EXIOBASE:
        {
            ‘(1)’: {“rd_embod”: ‘ln_rd_embod’},
            ‘(2)’: {‘rd_D’: ‘ln_rd_embod_D’, “rd_F”: ‘ln_rd_embod_F’},
            ‘(3)’: {‘rd_X’: ‘ln_rd_embod_X’, “rd_I”: ‘ln_rd_embod_I’},
            ‘(4)’: {
                ‘same_same’: ‘ln_rd_same_sector_same_country_embo_exio_mapF’,
                ‘same_diff’: ‘ln_rd_same_sector_diff_country_embo_exio_mapF’,
                ‘diff_same’: ‘ln_rd_diff_sector_same_country_embo_exio_mapF’,
                ‘diff_diff’: ‘ln_rd_diff_sector_diff_country_embo_exio_mapF’,
            },
        }
    """
    controls = controls or []
    rows = {}

    for sector in df_init["Sector"].unique():
        df_sec = df_init[df_init["Sector"] == sector].copy()
        df_sec = df_sec.set_index(["Region", "annee"]).sort_index()

        row = {}
        for model_name, var_map in models.items():
            exp_var = list(var_map.values()) + controls

            with contextlib.redirect_stdout(io.StringIO()):
                res, _ = estimation(data=df_sec, dep_var=dep_var, EXP_VAR=exp_var)

            if res is None:
                row[f"{model_name} Ck-wl"] = "-"
                for display_name in var_map:
                    row[f"{model_name} {display_name}"] = "-"
                continue

            row[f"{model_name} Ck-wl"] = _fmt_cell(
                res.params.get("C_k-w_l"), res.pvalues.get("C_k-w_l"), decimals)

            for display_name, ln_col in var_map.items():
                if ln_col in res.params.index:
                    row[f"{model_name} {display_name}"] = _fmt_cell(
                        res.params[ln_col], res.pvalues[ln_col], decimals)
                else:
                    row[f"{model_name} {display_name}"] = "-"

        rows[sector] = row

    table = pd.DataFrame(rows).T
    table.index.name = "Secteur"
    table.to_excel(xlsx_path)
    print(f"Tableau exporte : {xlsx_path}")
    return table

In [6]:
# Specifications exported in each table: (1) total R&D, (2) domestic/foreign, (3) inter/intra-sectoral, (4) the four channels
SPECS = {
    "(1)": {f"rd_{METRIC_WANTED}": f"ln_rd_{METRIC_WANTED}"},
    "(2)": {"rd_D": f"ln_rd_{METRIC_WANTED}_D", "rd_F": f"ln_rd_{METRIC_WANTED}_F"},
    "(3)": {"rd_X": f"ln_rd_{METRIC_WANTED}_X", "rd_I": f"ln_rd_{METRIC_WANTED}_I"},
    "(4)": {
        "same_same": f"ln_rd_same_sector_same_country_{METRIC}",
        "same_diff": f"ln_rd_same_sector_diff_country_{METRIC}",
        "diff_same": f"ln_rd_diff_sector_same_country_{METRIC}",
        "diff_diff": f"ln_rd_diff_sector_diff_country_{METRIC}",
    },
    "(5)": {"rd_x_int_propre": f"int_x_ln_rd_{METRIC_WANTED}_propre"},  # (R/Y) * ln(X) seul
}


In [7]:
# Estimate the full sample once for the selected base, metric, and run.
print(f"##### run {RUN_DIR} | {BASE} | {METRIC} | full sample #####")
input_path = panel_path()
if not os.path.exists(input_path):
    raise FileNotFoundError(f"Final regression panel not found: {input_path}")

df_init = load_panel()
out_path = xlsx_path_for()
os.makedirs(os.path.dirname(out_path), exist_ok=True)
tables = {
    "all": export_estimation_table(
        df_init, models=SPECS, controls=["2009_crisis"],
        xlsx_path=out_path,
    )
}


##### run SRDQ (c,s) | EXIOBASE | rd_dir_CI | full sample #####


C:\Users\carpc\.virtualenvs\ENTICE_T3.4-d6jcyxub\Lib\site-packages\pandas\core\arraylike.py:402: RuntimeWarning: invalid value encountered in log
  result = getattr(ufunc, method)(*inputs, **kwargs)


Tableau exporte : D:/ENTICE/runs/SRDQ (c,s)/estimation_results/EXIOBASE/tab_EXIOBASE_rd_dir_CI.xlsx
